# Deploying NVIDIA Nemotron 3.5 Lightning with SGLang

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint with SGLang on a single H100.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models and vision language models.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4)

**This notebook runs our recommended configuration: the NVFP4 checkpoint on a single H100 with DSpark speculative decoding.** Commands for the other tested configurations are at the end, under **Additional configurations**.

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- Python 3.10+
- Docker 

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint on a single H100 using SGLang
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget
- **Reference commands** for BF16, B200, DGX Spark, and the other speculative decoding methods

## Table of Contents

1. **Decoding options for this model** - Base, MTP, DFlash, and DSpark
2. **Environment setup** - Container image, client dependencies, and GPU check
   - Launch on NVIDIA Brev
   - Pull the SGLang Docker image
   - Install notebook client dependencies
   - Verify GPU
3. **OpenAI-compatible server** - Launch SGLang and confirm it is ready
   - Launch the Docker container
   - Configuration reference
   - Start server
   - Wait for the server to be ready
4. **Generate responses** - Chat completions, reasoning, and tool calling
   - Client setup
   - Single completion
   - Sequential completions
   - Streamed generation
   - Reasoning
   - Tool calling
   - Controlling reasoning budget
5. **Cleanup and shutdown** - Free the GPU and reset the kernel
6. **Additional configurations** - Reference commands for other hardware and precisions
   - 1x H100
   - 1x B200
   - 1x DGX Spark


## Decoding options for this model

Nemotron 3.5 Lightning can produce tokens four ways. All four serve the same weights and differ only in how many tokens come out of a single forward pass. The base option decodes one token per pass - the other three add speculative decoding, where a cheap draft proposes several tokens ahead, the model verifies them all in a single pass, and every token up to the first mismatch is kept. A rejected token invalidates itself and everything after it, so the payoff depends on how often drafts are right - a bad guess costs compute without producing output.

| Option | Where drafts come from | Extra weights to download |
|---|---|---|
| **Base** (no speculative decoding) | nothing, one token per pass | none |
| **MTP** | a prediction layer inside the checkpoint | none |
| **DFlash** | a separate block-diffusion draft model, a whole block per pass | DFlash checkpoint |
| **DSpark** | a separate semi-autoregressive draft model, a whole block per pass | DSpark checkpoint |

Pick exactly one: a server has a single draft path, so the flags conflict at launch rather than stacking.

**Concurrency decides whether speculation pays off.** With few requests in flight the GPU has spare capacity, and speculation spends it to shorten the critical path, which lowers per-request latency. Under heavy load the GPU is already busy producing real tokens, so verifying drafts that end up rejected takes throughput away from queued work. The commands here leave concurrency at the server default and only bound decode CUDA graphs with `--cuda-graph-max-bs-decode`, so benchmark the base configuration before assuming a speculative one wins under load.

**Draft length is the main knob.** MTP guesses with `--speculative-num-steps` and verifies `--speculative-num-draft-tokens`. DFlash and DSpark each take a block size instead, at values validated per configuration below, but the two flags count differently: `--speculative-dflash-block-size` is the verify width, one more than the draft depth, while `--speculative-dspark-block-size` is the draft depth itself. Guessing further ahead improves the best case per step but lowers the odds that the whole run is accepted, and wastes more compute when it is not.

**Two caches share the GPU.** Attention layers use a KV cache that grows with sequence length, while the Mamba layers keep a fixed-size recurrent state per sequence, halved here by `--mamba-ssm-dtype float16`. Both scale with `--context-length`, and `--mem-fraction-static` bounds how much of the GPU they may claim in total.

This notebook uses **DSpark**.

## Environment setup

### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Havtx6fAxwxMw9pLml5U6qAFrj)

### Pull the SGLang Docker image

The model runs inside an SGLang container. Pull it once before starting the server:

```shell
docker pull lmsysorg/sglang:dev-nemotron3-5-lightning
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `jinja2` renders the chat template it applies.

In [1]:
# Bootstrap pip only if the kernel environment is missing it
import importlib.util, subprocess, sys

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

%pip install -q openai==2.38.0 transformers==5.9.0 "jinja2>=3.1.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Verify GPU

Confirm your GPU is visible on the host before starting the Docker container.

> **Expected output:** One row per GPU, showing an H100 with roughly 80 GB of memory alongside the host driver version. If `nvidia-smi` is not found, the NVIDIA driver is not installed.

In [2]:
# Confirm the GPU is visible on the host
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

index, name, memory.total [MiB], driver_version
0, NVIDIA H100 PCIe, 81559 MiB, 570.148.08


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using SGLang.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the SGLang container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it \
  --gpus all \
  --cap-add SYS_NICE \
  --ipc=host \
  --network=host \
  --shm-size=16g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e SAFETENSORS_FAST_GPU=1 \
  --entrypoint /bin/bash \
  lmsysorg/sglang:dev-nemotron3-5-lightning
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

Run the `sglang serve` command below from inside this container. The commands in **Additional configurations** at the end run here too.

### Configuration reference

Settings for the configuration this notebook runs.

| Setting | NVFP4 | Why |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` | 4-bit weights fit a 30B MoE on one GPU |
| **Draft model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark` | Proposes the token blocks the model verifies |
| **Served model name** | `nemotron-3.5-lightning` | What the client cells send |
| **Hardware (this notebook)** | 1x H100 80GB | What these values were tuned on |
| **Docker image** | `lmsysorg/sglang:dev-nemotron3-5-lightning` | Pre-release build with day-0 model support |
| **MoE runner backend** | `marlin` (auto-selected on H100) | NVFP4 expert path on H100 |
| **Mamba SSM cache** | FP16 | Halves recurrent state per sequence |
| **Static memory fraction** | 0.85 | Caps memory for weights and caches |
| **Decode CUDA graph max batch** | 16 | Bounds CUDA graph memory |
| **Context length** | 1048576 (1M tokens) | Full context window |
| **Speculative decoding** | DSpark, block size 3 | Lower latency at this concurrency |
| **Reasoning parser** | `nemotron_3` | Splits thinking from the final answer |
| **Tool parser** | `qwen3_coder` | Turns tool syntax into OpenAI `tool_calls` |
| **Host / port** | `127.0.0.1:8000` | Local-only, matches the client cells |

### Start server

Run the command below from inside the Docker container terminal. This is the configuration the rest of the notebook is written against: the NVFP4 checkpoint on one H100 with DSpark speculative decoding.

> **Note:** The first launch takes a while. The progress readout can sit at `0%` for several minutes at a time: first while the weights download from Hugging Face and load, then while the server compiles CUDA kernels. That is expected rather than a hang, so give it time instead of restarting. Later launches reuse the cached weights and compiled kernels and start much faster.

> **Note:** Parser names are backend-specific: SGLang's `nemotron_3` is `nemotron_v3` in vLLM and `nemotron-v3` in TensorRT-LLM.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --speculative-algorithm DSPARK \
  --speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
  --speculative-dspark-block-size 3 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

### Wait for the server to be ready

In a new terminal, poll `/v1/models`:

```shell
until curl -sf http://localhost:8000/v1/models | grep -q nemotron-3.5-lightning; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

Then check what the server is actually serving, and send one short request to confirm it generates:

```shell
curl -s http://localhost:8000/v1/models | python3 -m json.tool

curl -sf http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "nemotron-3.5-lightning", "prompt": "Hello", "max_tokens": 16}'
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then `Server is ready` once `nemotron-3.5-lightning` is listed by `/v1/models`. The model list shows that name as the `id`, along with the running context length in `max_model_len`. The final command returns a short JSON completion, confirming the model is loaded and generating.

## Generate responses

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, the reasoning trace consumed the entire budget before the model produced an answer, so raise `max_tokens`.

### Client setup

In [3]:
from openai import OpenAI

# Set this to the model you launched the server with
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"  # SGLang does not require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"OpenAI client configured to use server at: {BASE_URL}")
print(f"Using model: {SERVED_MODEL_NAME}")

OpenAI client configured to use server at: http://localhost:8000/v1
Using model: nemotron-3.5-lightning


### Single completion

In [4]:
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Give me 3 bullet points about SGLang."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)

Reasoning: Here's a thinking process:

1.  **Analyze User Request:**
   - User wants 3 bullet points about "SGLang"
   - I need to provide concise, accurate bullet points
   - SGLang is a specific technology/system, so I need to recall or infer what it is

2.  **Identify "SGLang":**
   - SGLang stands for "SwiftGL" or "SGLang"? Let me think.
   - Actually, SGLang is a framework for efficient language model serving. Wait, let me verify mentally.
   - There's "SGLang" which is "SwiftGL"? No.
   - Actually, SGLang is "SGLang: Efficient Language Model Serving" or similar. Let me recall properly.
   - I think SGLang is a fast language model inference and serving framework, often compared to vLLM, TensorRT-LLM, etc. It might be developed by the same people or related to PyTorch/Transformers.
   - Let me search my training data mentally: SGLang is an open-source framework for serving large language models with high throughput and low latency. It supports various backends, batching strategies,

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [5]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    resp = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=1024,
    )
    print(f"Q: {prompt}")
    print(f"A: {resp.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12** (since 12 × 12 = 144). 

Note: While -12 also squared equals 144, the symbol √144 typically refers to the principal (non-negative) square root, which is 12.

Q: What is the capital of France?
A: The capital of France is Paris.

Q: Explain quantum computing in simple terms.
A: Here’s a down-to-earth breakdown:

### Classical vs. Quantum
- **Classical computers** use **bits**: each is either **0** or **1**. Think of it like a light switch: on or off.
- **Quantum computers** use **qubits** (quantum bits). Because of quantum physics, a qubit can be **0, 1, or both at the same time**. It’s like a spinning coin: while it’s spinning, it’s effectively both heads and tails until it lands and stops.

### Two Key Quantum Tricks
1. **Superposition** = The "both-at-once" state. A system of just 2 qubits can represent four possibilities simultaneously (00, 01, 10, 11). Add 10 qubits, and you’re handling ~1,000 states at once. Wit

### Streamed generation

In [6]:
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning_content", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Reasoning: Here, the user is asking for the first 5 prime numbers. I need to recall or compute the sequence of prime numbers. Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and itself. The sequence starts: 2, 3, 5, 7, 11, 13, ... So the first 5 are 2, 3, 5, 7, 11. I should provide them in order. I'll answer simply.

Content: The first 5 prime numbers are: 2, 3, 5, 7, 11.

### Reasoning

> **Note:** The model supports two modes: Reasoning ON (default) and Reasoning OFF. Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`, as shown below.

In [12]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a simple haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about SGLang."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False, "force_nonempty_content": True}}
)
print("Content:", resp.choices[0].message.content)

Reasoning on
Reasoning: Here's a thinking process:

1.  **Analyze User Input**: 
   - Request: "Write a simple haiku about GPUs."
   - Constraints: Haiku format (5-7-5 syllables), topic: GPUs (Graphics Processing Units).

2.  **Understand Haiku Structure**:
   - Line 1: 5 syllables
   - Line 2: 7 syllables
   - Line 3: 5 syllables

3.  **Brainstorm GPU-related concepts**: 
   - Graphics, cores, render, pixels, speed, compute, chips, games, displays, parallel, power, fans, silicon, vertices, shaders.

4.  **Drafting - Attempt 1**:
   - Silicon chips (5)
   - Render pixels quickly (5? "Ren-der pix-els quick-ly" = 6? Let's count: Ren(1)der(2) pix(3) els(4) quick(5) ly(6) -> 6. Need 7.)
   - Let's try: Silicon chips glow (5) - Sil-i-con chips glow (5? Sil(1)i(2)con(3) ch(4)ips(5) glow(6) -> 6. Hmm.)
   - Let's systematically count.

   Let me draft properly:
   Line 1 (5 syllables): 
   - "Silicon chips hum" -> Sil(1)i(2)con(3) ch(4)ips(5) hum(6) -> 6. Too many.
   - "Chip electrons" -> Ch

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`, then run the function and hand its result back so the model can answer the user.

> **Note:** When tool calling with reasoning enabled, pass `"force_nonempty_content": true` inside `chat_template_kwargs`. Without it, `content` can come back empty and the server may not surface the reasoning trace and the tool call together - coding agents in particular expect text alongside the call.

In [13]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calculate the tip amount for a bill",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                "tip_percentage": {"type": "integer", "description": "The percentage of tip to apply"},
            },
            "required": ["bill_total", "tip_percentage"],
        },
    },
}]

messages = [{"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "force_nonempty_content": True}},
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Content:", choice.message.content)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage` as integers.
   - `bill_total` = 50
   - `tip_percentage` = 15

3.  **Check Tool Parameters:**
   - `bill_total`: integer, required
   - `tip_percentage`: integer, required
   - Both match the user's input.

4.  **Call the Tool:**
   - Invoke `calculate_tip` with `bill_total=50`, `tip_percentage=15`.

5.  **Formulate Response:**
   - After getting the result, I'll state the tip amount and possibly the total.
   - Wait, I need to actually call the function first to get the result, or I can just compute it mentally, but the prompt says "You have access to the following functions" and expects me to use them. I'll call it.

Let's call the function.⟩

Content: 

Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_a0

In [14]:
import json

def calculate_tip(bill_total, tip_percentage):
    """Return the tip and the final total for a bill."""
    tip = round(bill_total * tip_percentage / 100, 2)
    return {"tip": tip, "total": round(bill_total + tip, 2)}

# Map schema names to real functions, then run whichever one the model picked
tool_functions = {"calculate_tip": calculate_tip}

call = choice.message.tool_calls[0]
result = tool_functions[call.function.name](**json.loads(call.function.arguments))

# Hand the result back so the model can answer the user
followup = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages + [
        choice.message,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ],
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Tool result:", result)
print("Final answer:", followup.choices[0].message.content)

Tool result: {'tip': 7.5, 'total': 57.5}
Final answer: A 15% tip on a $50 bill comes out to **$7.50**, making the total amount **$57.50**.


### Controlling reasoning budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [ ]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [17]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here's a thinking process:

1.  **Analyze the Request:**
   - User wants a haiku about GPUs.
   - Haiku structure: 5 syllables, 7 syllables, 5 syllables (total 17 syllables).
   - Topic: GPUs (Graphics Processing Units).

2.  **Brainstorm GPU-related concepts:**
   - Graphics, cores, chips, parallel processing, gaming, rendering, AI, deep learning, matrices, volts, fans, speed, cores, parallel.

3.  **Drafting - Attempt 1:**
   GPUs blaze (5) - Need to count carefully.
.
Content: GPUs blaze bright,
Parallel cores render the night,
Speed in silicon.


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the SGLang server in the terminal where it was started (`Ctrl+C`).
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Restart the kernel if needed to ensure a clean state.

## Additional configurations

Reference commands for the configurations this notebook does not run.

Each entry gives a full base command followed by the flags that switch on a speculator. Add a flag block to the base command - the blocks are fragments and do not run on their own. MTP's draft head lives inside the target checkpoint, so its draft path is that checkpoint itself - DFlash and DSpark always draft from the NVFP4 checkpoints, even when the target is BF16.

> **Note:** These commands set a 1M-token context window, except BF16 on H100, which is 256K to fit in 80GB. Set `--context-length` lower if you are memory-constrained or want more KV-cache headroom at high concurrency.

### 1x H100

#### NVFP4

**Base**

On Hopper the NVFP4 weights run through W4A16 kernels, and the `flashinfer` Mamba backend is not required since FA3 target attention is selected by default.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

**Add MTP**

> **Note:** Keep `--cuda-graph-max-bs-decode 16`. Without it, MTP's decode graphs reserve enough memory that short requests pass but a large-token request hits an out-of-memory error.

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 4
```

**Add DSpark**: this is the primary path above, under **Start server**.

#### BF16

**Base**

This sets a 256K context window, which is what fits one 80GB H100 in BF16.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 262144 \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

> **Note:** BF16 weights take roughly 60GB of an 80GB H100, leaving little room for a draft model. Benchmark before assuming a speculator wins here.

**Add MTP**

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 4
```

**Add DSpark**

```shell
--speculative-algorithm DSPARK \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative-dspark-block-size 3
```

### 1x B200

#### NVFP4

**Base**

Balanced baseline with no speculative decoding.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-backend flashinfer \
  --mamba-ssm-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

**Add MTP**

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 6
```

**Add DSpark**

```shell
--speculative-algorithm DSPARK \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative-dspark-block-size 3
```

#### BF16

**Base**

Balanced baseline with no speculative decoding, at a 1M-token context window. Set `--context-length` lower if you are memory-constrained or want more KV-cache headroom at higher concurrency.

```shell
SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1 sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-backend flashinfer \
  --mamba-ssm-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

**Add MTP**

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 6
```

**Add DSpark**

```shell
--speculative-algorithm DSPARK \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative-dspark-block-size 3
```

### 1x DGX Spark

#### NVFP4

**Base**

The memory budget is smaller than an H100's, so `--mem-fraction-static` and `--cuda-graph-max-bs-decode` are both lowered.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.78 \
  --cuda-graph-max-bs-decode 4 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

**Add MTP**

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 4
```

**Add DSpark**

```shell
--speculative-algorithm DSPARK \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative-dspark-block-size 3
```

#### BF16

**Base**

Same settings as NVFP4 on this hardware - only the checkpoint changes.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --context-length 1048576 \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.78 \
  --cuda-graph-max-bs-decode 4 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

**Add MTP**

```shell
--speculative-algorithm EAGLE \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
--speculative-num-steps 5 \
--speculative-eagle-topk 1 \
--speculative-num-draft-tokens 6
```

**Add DFlash**

```shell
--speculative-algorithm DFLASH \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative-dflash-block-size 4
```

**Add DSpark**

```shell
--speculative-algorithm DSPARK \
--speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative-dspark-block-size 3
```
